# 청크 사이즈 A/B — RAGAS (context recall / precision)

히스토그램·길이 통계([chunk_size_test.ipynb](chunk_size_test.ipynb))는 **사전 필터**일 뿐, 확정 근거가 아니다.
여기서는 **검색 품질**로 실측한다: `800/120` vs `1000/150` 두 후보만 놓고
**context recall / context precision** 을 A/B 로 비교한다.

검색 방식도 변수라서 3가지를 함께 본다 (사용자 관찰: **BM25는 큰 청크(1000), dense는 focused한 청크**를 선호):
- **dense** — KURE-v1 코사인 (실제 서비스가 쓰는 방식, `services/retrieval/search.py`)
- **bm25** — Okapi BM25 (희소 검색)
- **hybrid** — 위 둘을 RRF(Reciprocal Rank Fusion)로 융합

→ 결과 표는 **`config × 검색방식`** 그리드다. 어느 청크 사이즈가 어떤 검색기에서 이기는지 실측으로 갈린다.

### ⚠️ 실행 전 알아둘 것
- **전용 커널로 실행** — 이 노트북은 반드시 Jupyter 커널 **`Python (SKN eval / ragas)`** 로 실행한다(아래 0번 참고). 앱 `.venv` 커널로 돌리면 `import ragas` 가 깨진다.
- **OpenAI API 키 필요** — RAGAS 는 LLM 심판을 쓴다(+ 평가셋 자동 생성). `.env` 의 `OPENAI_API_KEY` 사용, 토큰 비용 발생.
- **KURE-v1 임베딩(CPU)은 느리다** — 청크 수천 개 × 2 설정. GPU 면 수 분, CPU 면 십수 분+. `LIMIT_PAGES` 로 줄여 먼저 스모크 테스트하라.
- **평가셋은 코퍼스에서 자동 생성**해 `test/ragas_testset.jsonl` 로 캐시한다. **반드시 사람이 검수/보정**한 뒤 본 평가를 돌려라 — 정답이 틀리면 recall 점수가 오염된다.

## 0. 커널 확인 — 전용 eval 환경

RAGAS(0.3.9)는 **langchain 0.3 라인**을 요구하는데, 앱의 **langgraph 1.x 는 langchain 1.x** 를 요구해 한 venv 에 공존이 안 된다.
그래서 평가 전용 venv `backend/.venv-eval` 에 호환 스택을 깔고 Jupyter 커널로 등록해 두었다.

**이 노트북 우측 상단에서 커널을 `Python (SKN eval / ragas)` 로 선택**하고 아래 셀을 실행하라.
앱 `.venv` 커널이면 아래 셀에서 `import ragas` 가 실패한다.

> eval venv 를 다시 만들려면 (`backend/` 에서):
> ```bash
> uv venv .venv-eval --python 3.12
> uv pip install --python .venv-eval "ragas>=0.2,<0.4" >   "langchain-core>=0.3,<0.4" "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" >   "langchain-openai>=0.2,<0.4" rank-bm25 openai python-dotenv numpy ipykernel "sentence-transformers>=3.0"
> .venv-eval/Scripts/python -m ipykernel install --user --name skn-eval --display-name "Python (SKN eval / ragas)"
> ```

In [ ]:
# 커널이 맞는지 확인. 아래 import 가 실패하면 커널을 'Python (SKN eval / ragas)' 로 바꾸세요.
import importlib.metadata as md
for p in ["ragas", "langchain-core", "langchain-community", "langchain-openai", "rank-bm25", "sentence-transformers"]:
    try:
        print(f"  {p:22} {md.version(p)}")
    except Exception as e:
        print(f"  {p:22} 없음 ({e})  ← eval 커널이 아닙니다")

from ragas import EvaluationDataset, evaluate   # 여기서 에러나면 커널이 틀린 것
assert md.version("langchain-core").startswith("0.3"), "langchain-core 가 0.3 라인이 아닙니다 — eval 커널을 쓰세요."
print("OK — eval 커널 확인됨 (langchain 0.3 라인 + ragas)")

## 1. 준비 — 경로 · API 키 · 로직 import · KURE 로드

In [ ]:
import os, sys, json, time, random, re
from pathlib import Path
import numpy as np
from dotenv import load_dotenv

# backend/ 를 sys.path 에 올린다 (test.ingest_pdf 재사용).
BACKEND = Path.cwd()
while not (BACKEND / "test" / "ingest_pdf.py").exists():
    if BACKEND == BACKEND.parent:
        raise RuntimeError("backend/ 를 찾지 못했습니다. 노트북을 backend/test 에서 여세요.")
    BACKEND = BACKEND.parent
sys.path.insert(0, str(BACKEND))
load_dotenv(BACKEND / ".env")

from test.ingest_pdf import clean_pdf_text, build_chunks, DEFAULT_DATA_DIR, DEFAULT_MIN_PAGE_CHARS
from test.ingest_kure import iter_records, resolve_data_dir, collect_files, load_model

print("backend    :", BACKEND)
print("openai key :", "OK" if os.getenv("OPENAI_API_KEY") else "--- 없음 (평가셋 생성·RAGAS 불가)")

# 비교할 두 후보만.
CONFIGS = {"800/120": (800, 120), "1000/150": (1000, 150)}
LIMIT_PAGES = None   # 스모크 테스트: 예) 60 으로 두면 페이지 60개만 사용(빠름). 본 평가는 None.
TESTSET_PATH = BACKEND / "test" / "ragas_testset.jsonl"

print("KURE-v1 로드 중… (최초 실행 시 ~2GB 다운로드)")
model = load_model(None)     # device auto
print("KURE-v1 준비 완료")

## 2. 데이터 로드 + 전처리 + 두 후보로 청킹

In [ ]:
DATA_DIR = resolve_data_dir(DEFAULT_DATA_DIR)
files = collect_files(DATA_DIR, "")

pages = []
for path in files:
    for rec in iter_records(path):
        pages.append({"content": clean_pdf_text(rec["content"]),
                      "metadata": rec["metadata"], "file": path.name})
if LIMIT_PAGES:
    random.seed(0)
    pages = random.sample(pages, min(LIMIT_PAGES, len(pages)))
print(f"페이지: {len(pages)}")

# config 별 청크 코퍼스.
corpora = {}
for name, (size, overlap) in CONFIGS.items():
    chunks = []
    for pg in pages:
        chunks += build_chunks({"content": pg["content"], "metadata": pg["metadata"]},
                               size, overlap, DEFAULT_MIN_PAGE_CHARS)
    corpora[name] = chunks
    print(f"  {name:>9} → 청크 {len(chunks)}")

## 3. 인덱스 — dense(KURE) + sparse(BM25)

**가장 느린 셀.** config 별로 모든 청크를 KURE 로 임베딩한다. `embed_text`(문서 제목·페이지 맥락 헤더 포함)를
임베딩해 실제 색인과 동일하게 맞춘다. BM25 는 한국어 형태소 분석기 없이 간이 토크나이저(어절+한글 바이그램)를 쓴다
— 더 정확히 하려면 `kiwipiepy` 로 교체하라.

In [ ]:
_TOK = re.compile(r"[0-9A-Za-z]+|[가-힣]+")

def tokenize(text: str) -> list[str]:
    """BM25용 간이 한국어 토크나이저: 영숫자 어절 + 한글은 문자 바이그램."""
    toks = []
    for m in _TOK.findall(text):
        if "\uac00" <= m[0] <= "\ud7a3" and len(m) >= 2:
            toks += [m[i:i+2] for i in range(len(m) - 1)]   # 한글 바이그램
        else:
            toks.append(m.lower())
    return toks or ["_"]

from rank_bm25 import BM25Okapi

indexes = {}
for name, chunks in corpora.items():
    t = time.perf_counter()
    embs = model.encode([c["embed_text"] for c in chunks],
                        batch_size=64, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=True)
    bm25 = BM25Okapi([tokenize(c["content"]) for c in chunks])
    indexes[name] = {"emb": np.asarray(embs, dtype=np.float32), "bm25": bm25, "chunks": chunks}
    print(f"  {name:>9} 임베딩+BM25 완료 ({time.perf_counter()-t:.1f}s, dim={embs.shape})")

## 4. 검색기 — dense / bm25 / hybrid(RRF)

In [ ]:
RRF_POOL = 50   # 융합 전에 각 방식에서 뽑아 둘 후보 수

def _dense_rank(name, query, pool):
    q = np.asarray(model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0], np.float32)
    sims = indexes[name]["emb"] @ q          # 정규화돼 있으니 내적 = 코사인
    return np.argsort(-sims)[:pool].tolist()

def _bm25_rank(name, query, pool):
    scores = indexes[name]["bm25"].get_scores(tokenize(query))
    return np.argsort(-scores)[:pool].tolist()

def _rrf(ranked_lists, k=60, topn=None):
    score = {}
    for lst in ranked_lists:
        for rank, idx in enumerate(lst):
            score[idx] = score.get(idx, 0.0) + 1.0 / (k + rank + 1)
    order = sorted(score, key=score.get, reverse=True)
    return order[:topn] if topn else order

def retrieve(name, mode, query, k=8):
    if mode == "dense":
        idxs = _dense_rank(name, query, k)
    elif mode == "bm25":
        idxs = _bm25_rank(name, query, k)
    elif mode == "hybrid":
        idxs = _rrf([_dense_rank(name, query, RRF_POOL), _bm25_rank(name, query, RRF_POOL)], topn=k)
    else:
        raise ValueError(mode)
    return [indexes[name]["chunks"][i]["content"] for i in idxs]

MODES = ["dense", "bm25", "hybrid"]
TOP_K = 8
# 눈으로 한 번 확인
demo = retrieve("1000/150", "hybrid", "계약갱신 시 임대료를 5% 넘게 올릴 수 있나요?", k=3)
for i, c in enumerate(demo):
    print(f"[hybrid #{i}] {c[:120]}…\n")

## 5. 평가셋 — 코퍼스에서 자동 생성 (+ 캐시)

`test/ragas_testset.jsonl` 이 있으면 로드, 없으면 코퍼스 청크를 표본으로 **질문+정답(reference)** 을 LLM 으로 생성한다.
정답을 청크 내용에 **엄격히 근거**하도록 프롬프트한다(환각 방지). **생성 후 사람이 검수/보정**하라.

In [ ]:
N_QUESTIONS = 12    # 스모크는 6~8, 본 평가는 30+ 권장

def _load_testset(path):
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]

if TESTSET_PATH.exists():
    testset = _load_testset(TESTSET_PATH)
    print(f"평가셋 로드: {len(testset)}건 ← {TESTSET_PATH}")
else:
    from openai import OpenAI
    client = OpenAI()
    # 1000/150 코퍼스에서 source_type 골고루 표본(정보량 있는 긴 청크 우선)
    pool = [c for c in corpora["1000/150"] if len(c["content"]) >= 300]
    random.seed(0); random.shuffle(pool)
    picks = pool[:N_QUESTIONS]
    SYS = ("너는 한국 전월세(주택임대차) 법률 QA 데이터셋 제작자다. 주어진 '문서 조각'만 근거로 "
           "사용자가 실제로 물을 법한 질문 1개와, 그 조각에서 직접 확인되는 간결한 정답을 만든다. "
           "조각에 없는 사실은 절대 지어내지 마라. JSON 으로만 답하라: "
           '{"question": "...", "reference": "..."}')
    testset = []
    for c in picks:
        r = client.chat.completions.create(
            model="gpt-4o-mini", temperature=0.2,
            response_format={"type": "json_object"},
            messages=[{"role": "system", "content": SYS},
                      {"role": "user", "content": "문서 조각:\n" + c["content"][:1500]}])
        try:
            obj = json.loads(r.choices[0].message.content)
            if obj.get("question") and obj.get("reference"):
                testset.append({"question": obj["question"], "reference": obj["reference"],
                                "source_type": c["metadata"].get("source_type")})
        except Exception as e:
            print("  건너뜀:", e)
    TESTSET_PATH.write_text("\n".join(json.dumps(t, ensure_ascii=False) for t in testset), encoding="utf-8")
    print(f"평가셋 {len(testset)}건 생성 → {TESTSET_PATH}  (검수 후 다시 실행하면 이 파일을 그대로 씁니다)")

for t in testset[:3]:
    print("Q:", t["question"]); print("A:", t["reference"][:120], "…\n")

## 6. RAGAS A/B — context recall & precision

각 `(config, mode)` 조합으로 질문마다 top-k 청크를 검색해 RAGAS 로 채점한다.
- **context recall** — 정답(reference)의 내용이 검색된 청크로 뒷받침되는 비율. **낮으면 = 필요한 근거를 놓침**(청킹이 너무 잘게 쪼갰거나 너무 뭉쳐 회수 실패).
- **context precision** — 관련 청크가 상위에 랭크됐는지. **낮으면 = 관련 없는 청크가 위로**(노이즈).

In [ ]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

judge = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
metrics = [LLMContextRecall(), LLMContextPrecisionWithReference()]

def eval_one(name, mode):
    samples = [{
        "user_input": t["question"],
        "retrieved_contexts": retrieve(name, mode, t["question"], k=TOP_K),
        "reference": t["reference"],
    } for t in testset]
    res = evaluate(dataset=EvaluationDataset.from_list(samples), metrics=metrics, llm=judge)
    df = res.to_pandas()
    nums = df.select_dtypes("number").mean()
    # 컬럼명이 버전마다 조금 달라서 부분일치로 집는다.
    def pick(key):
        for c in nums.index:
            if key in c:
                return round(float(nums[c]), 3)
        return float("nan")
    return {"recall": pick("recall"), "precision": pick("precision")}

results = {}
for name in CONFIGS:
    for mode in MODES:
        print(f"평가 중: {name:>9} / {mode:<6} …")
        results[(name, mode)] = eval_one(name, mode)

# ── 비교표 ──
print("\n" + "=" * 62)
print(f"{'config':>10} {'mode':>8} {'ctx_recall':>12} {'ctx_precision':>14}")
print("-" * 62)
for name in CONFIGS:
    for mode in MODES:
        r = results[(name, mode)]
        print(f"{name:>10} {mode:>8} {r['recall']:>12} {r['precision']:>14}")
print("=" * 62)

## 7. 해석 가이드

- **후보를 고르는 법**: 같은 `mode` 안에서 `800/120` vs `1000/150` 의 recall/precision 을 비교하라. 서비스가 **dense** 중심이면 dense 행을, hybrid 도입을 검토 중이면 hybrid 행을 기준으로.
- **사용자 관찰 검증**: 보통 **bm25 는 1000/150**(큰 청크에 키워드가 몰림)에서, **dense 는 800/120**(focused)에서 상대적으로 유리하게 나오는지 확인하라. 갈리면 → **hybrid 가 어느 사이즈에서 두 장점을 가장 잘 합치는지**가 결정 포인트다.
- **recall ↔ precision 트레이드오프**: 큰 청크는 recall↑ precision↓ 경향(한 청크에 관련·무관 정보 혼재), 작은 청크는 그 반대. 서비스 답변 품질에 어느 쪽이 더 치명적인지로 가중치를 둬라.
- **신뢰도 주의**: 질문 수가 적으면 점수가 흔들린다. `N_QUESTIONS` 를 30+ 로 늘리고 **평가셋을 검수**한 뒤 값을 신뢰하라. `TOP_K`·`RRF_POOL`·`min_score` 도 결과를 바꾸는 변수다.
- **재현성**: 확정 전, 검수한 `ragas_testset.jsonl` 을 커밋해 같은 셋으로 재평가할 수 있게 하라.